In [5]:
!apt-get update -y
!apt-get install -y curl
!pip install pandas scikit-learn regex

# --- Install Ollama (optional; may not work on Colab GPU runtimes) ---
!curl -fsSL https://ollama.com/install.sh | sh || echo "⚠️ Ollama install may not work in Colab (requires local environment)"


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,179 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os, re, subprocess, json, time
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ===============================================================
# Helper: Run classification with Ollama (Gemma3)
# ===============================================================
def classify_news(article_text: str):
    """Run the Arabic news classification using Gemma3 via Ollama."""
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(5)
        subprocess.run(["ollama", "pull", "gemma3:4b"], check=True)
    except Exception as e:
        print("⚠️ Ollama may not be available:", e)
        return {
            "Topic": "unknown",
            "Reality": "unknown",
            "Confidence": "0%",
            "Main_reason_for_prediction": "Ollama unavailable"
        }

    prompt = f"""
    You are a multilingual Arabic news analysis assistant.
    Your job is twofold:
    1. Determine the **main topic** of the Arabic article from this list, you can say unknown if only the text is not found:
        [
        "Arts",
        "Crime",
        "Disaster and Accident",
        "Economy",
        "Education",
        "Environment",
        "Health",
        "Human Interest",
        "Labour",
        "Lifestyle and Leisure",
        "Politics",
        "Religion and Belief",
        "Science and Technology",
        "Society",
        "Sport",
        "War",
        "Weather",
        "Unknown"
        ]

    2. Determine if the article is **Real** or **Fake** using these metrics, if the text is not found say unknown:
        - Lexical richness
        - Sentiment exaggeration
        - Named entities
        - Clickbait words (صادم، لن تصدق، مفاجئ، خطير)
        - Logical and factual coherence
        - Consistency between title and body
        - Formal Arabic news tone

    Output format (strictly):
    [Topic_category, Reality, Confidence, Main_reason_for_prediction]

    Example:
    [Politics, Real, 91%, Factual reporting with verifiable named entities]

    Article:
    {article_text}
    """

    proc = subprocess.Popen(
        [
            "curl", "-s", "-X", "POST", "http://localhost:11434/api/generate",
            "-H", "Content-Type: application/json",
            "-d", json.dumps({
                "model": "gemma3:4b",
                "prompt": prompt,
                "options": {"temperature": 0}
            })
        ],
        stdout=subprocess.PIPE,
        text=True
    )

    raw_output = ""
    for line in proc.stdout:
        if '"response":"' in line:
            part = line.split('"response":"')[-1].split('"', 1)[0]
            raw_output += part

    raw_output = raw_output.replace("\\n", "\n").strip()
    match = re.search(r"\[(.*?)\]", raw_output)
    if match:
        values = [v.strip() for v in match.group(1).split(",")]
    else:
        values = []

    while len(values) < 4:
        values.append("")

    reality_val = values[1].lower()
    if reality_val in ["reality", "real", "true"]:
        reality = "Real"
    else:
        reality = values[1]  # keep original or set to "FAKE" as needed

    # Return dictionary
    return {
        "Topic": values[0],
        "Reality": reality,
        "Confidence": values[2],
        "Main_reason_for_prediction": values[3]
    }

# ===============================================================
# Batch Processor (per file)
# ===============================================================
def process_file(input_path, output_path):
    df = pd.read_csv(input_path)
    if "Article_Text" not in df.columns:
        raise ValueError("❌ Missing 'Article_Text' column")
    if "True_Label" not in df.columns:
        raise ValueError("❌ Missing 'True_Label' column (REAL/FAKE)")

    print(f"🚀 Classifying {input_path} ...")
    results = [classify_news(text) for text in df["Article_Text"]]
    results_df = pd.DataFrame(results)
    final_df = pd.concat([df, results_df], axis=1)

    columns_order = [
        "Title", "Link", "website", "Article_Text",
        "Topic", "Reality", "Confidence", "Main_reason_for_prediction", "True_Label"
    ]
    final_df = final_df[[c for c in columns_order if c in final_df.columns]]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved classified file → {output_path}")

    return final_df


# ===============================================================
# Run classification for all files and compute GLOBAL metrics
# ===============================================================
files = [
    ("/content/drive/MyDrive/labeled_data/labeled_elshrouk.csv", "/content/drive/MyDrive/classified_data/elshrouk_reality.csv"),
    ("/content/drive/MyDrive/labeled_data/labeled_elmasrielyoum.csv", "/content/drive/MyDrive/classified_data/elmasrielyoum_reality.csv"),
    ("/content/drive/MyDrive/labeled_data/labeled_elwatan.csv", "/content/drive/MyDrive/classified_data/elwatan_reality.csv"),
    ("/content/drive/MyDrive/labeled_data/labeled_masrawy.csv", "/content/drive/MyDrive/classified_data/masrawy_reality.csv"),
    ("/content/drive/MyDrive/labeled_data/labeled_youm7.csv", "/content/drive/MyDrive/classified_data/youm7_reality.csv"),
    ("/content/drive/MyDrive/labeled_data/labeled_test.csv", "/content/drive/MyDrive/classified_data/test_reality.csv")
]

all_data = []

for inp, outp in files:
    if os.path.exists(inp):
        df = process_file(inp, outp)
        all_data.append(df)
    else:
        print(f"⚠️ Skipping missing file: {inp}")

# ===============================================================
# Calculate combined accuracy, precision, recall, F1
# ===============================================================


🚀 Classifying /content/drive/MyDrive/labeled_data/labeled_elshrouk.csv ...
✅ Saved classified file → /content/drive/MyDrive/classified_data/elshrouk_reality.csv
🚀 Classifying /content/drive/MyDrive/labeled_data/labeled_elmasrielyoum.csv ...
✅ Saved classified file → /content/drive/MyDrive/classified_data/elmasrielyoum_reality.csv
🚀 Classifying /content/drive/MyDrive/labeled_data/labeled_elwatan.csv ...
✅ Saved classified file → /content/drive/MyDrive/classified_data/elwatan_reality.csv
🚀 Classifying /content/drive/MyDrive/labeled_data/labeled_masrawy.csv ...
✅ Saved classified file → /content/drive/MyDrive/classified_data/masrawy_reality.csv
🚀 Classifying /content/drive/MyDrive/labeled_data/labeled_youm7.csv ...
✅ Saved classified file → /content/drive/MyDrive/classified_data/youm7_reality.csv
🚀 Classifying /content/drive/MyDrive/labeled_data/labeled_test.csv ...
✅ Saved classified file → /content/drive/MyDrive/classified_data/test_reality.csv


In [8]:
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)

    # Filter to only REAL / FAKE
    valid_mask = combined_df["Reality"].isin(["Real", "Fake"])
    y_true = combined_df.loc[valid_mask, "True_Label"].str.upper().str.strip()
    y_pred = combined_df.loc[valid_mask, "Reality"].str.upper().str.strip()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label="REAL", zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label="REAL", zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label="REAL", zero_division=0)

    metrics = {
        "Total_Files": len(all_data),
        "Total_Samples": len(combined_df),
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1_Score": round(f1, 4)
    }

    metrics_path = "/content/drive/MyDrive/classified_data/overall_metrics_report.csv"
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")

    print("📊 ================================")
    print("📊 OVERALL MODEL PERFORMANCE")
    print("📊 ================================")
    print(metrics_df)
    print(f"✅ Global metrics saved to {metrics_path}")

else:
    print("⚠️ No valid files processed — skipping global metrics.")


📊 ================================
📊 OVERALL MODEL PERFORMANCE
📊 ================================
   Total_Files  Total_Samples  Accuracy  Precision  Recall  F1_Score
0            6            585    0.9181     0.9342  0.9416    0.9379
✅ Global metrics saved to /content/drive/MyDrive/classified_data/overall_metrics_report.csv
